In [1]:
!pip -q install -U transformers datasets accelerate sentencepiece sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 6.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [2]:
import torch
import transformers
import datasets
import sentencepiece
import sacrebleu

print("All imports successful!")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU available:", torch.cuda.is_available())

All imports successful!
PyTorch: 2.11.0+cu130
Transformers: 5.19.0
GPU available: True


In [3]:
import os
import torch
import pandas as pd

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

import sacrebleu

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
GPU available: True
GPU: Tesla T4


In [4]:
MODEL_NAME = "google-t5/t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

print("Model loaded successfully!")
print("Device:", device)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded successfully!
Device: cuda


In [5]:
def translate_text(text, source_language, target_language):
    prompt = (
        f"translate {source_language} to {target_language}: "
        f"{text}"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=128,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4
        )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )


print(
    translate_text(
        "Hello, how are you?",
        "English",
        "French"
    )
)

Bonjour, comment êtes-vous?


In [6]:
dataset = load_dataset(
    "Helsinki-NLP/opus_books",
    "en-fr"
)

print(dataset)
print(dataset["train"][0])

README.md:   0%|          | 0.00/28.1k [00:00<?, ?B/s]

en-fr/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

en-fr/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/127085 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 127085
    })
})
{'id': '0', 'translation': {'en': 'The Wanderer', 'fr': 'Le grand Meaulnes'}}


In [7]:
split_dataset = dataset["train"].train_test_split(
    test_size=0.10,
    seed=42
)

train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

# Keep a smaller validation set from the training data
validation_split = train_dataset.train_test_split(
    test_size=0.05,
    seed=42
)

train_dataset = validation_split["train"]
validation_dataset = validation_split["test"]

print("Training rows:", len(train_dataset))
print("Validation rows:", len(validation_dataset))
print("Testing rows:", len(test_dataset))

Training rows: 108657
Validation rows: 5719
Testing rows: 12709


In [8]:
SOURCE_LANG = "en"
TARGET_LANG = "fr"

SOURCE_LANGUAGE_NAME = "English"
TARGET_LANGUAGE_NAME = "French"

PREFIX = (
    f"translate {SOURCE_LANGUAGE_NAME} "
    f"to {TARGET_LANGUAGE_NAME}: "
)

MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 128

print("Prefix:", PREFIX)

Prefix: translate English to French: 


In [9]:
def preprocess_function(examples):
    inputs = [
        PREFIX + item[SOURCE_LANG]
        for item in examples["translation"]
    ]

    targets = [
        item[TARGET_LANG]
        for item in examples["translation"]
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_validation = validation_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=validation_dataset.column_names
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names
)

print(tokenized_train[0])

Map:   0%|          | 0/108657 [00:00<?, ? examples/s]

Map:   0%|          | 0/5719 [00:00<?, ? examples/s]

Map:   0%|          | 0/12709 [00:00<?, ? examples/s]

{'input_ids': [13959, 1566, 12, 2379, 10, 978, 9663, 7, 3510, 376, 5, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [679, 7, 2890, 9, 6715, 29, 7, 90, 3, 7, 670, 77, 5320, 5, 1]}


In [10]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

print("Data collator ready!")

Data collator ready!


In [11]:
OUTPUT_DIR = "/content/t5-translation-output"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,

    learning_rate=2e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    gradient_accumulation_steps=4,
    num_train_epochs=2,

    weight_decay=0.01,
    predict_with_generate=True,

    fp16=torch.cuda.is_available(),
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",
    seed=42
)

print("Training configuration ready!")

Training configuration ready!


In [12]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = predictions.tolist()

    labels = labels.copy()
    labels[labels == -100] = tokenizer.pad_token_id

    decoded_predictions = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True
    )

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    predictions_clean = [
        pred.strip() for pred in decoded_predictions
    ]

    references_clean = [
        ref.strip() for ref in decoded_labels
    ]

    bleu = sacrebleu.corpus_bleu(
        predictions_clean,
        [references_clean]
    )

    return {
        "bleu": bleu.score
    }

In [13]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer initialized successfully!")

Trainer initialized successfully!


In [16]:
def translate(text, source_language, target_language):
    prompt = (
        f"translate {source_language} to {target_language}: {text}"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4
        )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [17]:
text = "Hello, how are you today?"

result = translate(
    text,
    source_language="English",
    target_language="French"
)

print("Original:", text)
print("Translation:", result)

Original: Hello, how are you today?
Translation: Bonjour, comment êtes-vous aujourd'hui?


In [18]:
print(translate("Good morning", "English", "French"))
print(translate("How are you?", "English", "German"))
print(translate("Thank you", "English", "Romanian"))
print(translate("Hello", "English", "Tamil"))
print(translate("Hello", "English", "Hindi"))

Bonjour.
Wie sind Sie?
Vă mulţumesc.
Tamil: Hallo!
Hallo!


In [19]:
!pip -q install gradio

In [20]:
import gradio as gr

def run_translation(text, source, target):
    if not text.strip():
        return "Please enter text to translate."

    return translate(text, source, target)

demo = gr.Interface(
    fn=run_translation,
    inputs=[
        gr.Textbox(label="Text to translate"),
        gr.Dropdown(
            ["English", "French", "German", "Romanian", "Tamil", "Hindi"],
            value="English",
            label="Source language"
        ),
        gr.Dropdown(
            ["French", "German", "Romanian", "Tamil", "Hindi", "English"],
            value="French",
            label="Target language"
        )
    ],
    outputs=gr.Textbox(label="Translation"),
    title="T5-Base Translation App",
    description="Translation using the pretrained Google T5-Base model without fine-tuning."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c9665891193545591b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [23]:
import os

SAVE_PATH = "/content/t5-base-model"
os.makedirs(SAVE_PATH, exist_ok=True)

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!


In [24]:
import shutil

ZIP_PATH = shutil.make_archive(
    "/content/t5-base-model",
    "zip",
    root_dir="/content/t5-base-model"
)

print("ZIP created:", ZIP_PATH)

ZIP created: /content/t5-base-model.zip


In [25]:
from google.colab import files

files.download("/content/t5-base-model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>